In [ ]:
import json
from collections import defaultdict

baseline_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-v1-5/generate_images/pick_a_pic_v2/sd-v1-5/ckpt-0/evaluation_results.jsonl"
method_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-v1-5/generate_images/pick_a_pic_v2/top_512_images_no_anime_colorfulness_pickscore_002-hpdv3_all-inpainting-w_sft_0.5/ckpt-800/evaluation_results.jsonl"

baseline_data = {}
with open(baseline_reward_score_csv_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        baseline_data[data['sample_id']] = data

method_data = {}
with open(method_reward_score_csv_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        method_data[data['sample_id']] = data

print(f"baseline: {baseline_reward_score_csv_path}")
print(f"method: {method_reward_score_csv_path}")
print(f"Baseline samples: {len(baseline_data)}")
print(f"Method samples: {len(method_data)}")

In [ ]:
if baseline_data and method_data:
    common_sample_ids = set(baseline_data.keys()) & set(method_data.keys())
    print(f"Common sample IDs: {len(common_sample_ids)}")
    
    if common_sample_ids:
        first_id = list(common_sample_ids)[0]
        eval_models = list(baseline_data[first_id]['scores'].keys())
        print(f"Evaluation models: {eval_models}")
    else:
        print("Warning: No common sample IDs found!")
        eval_models = []
else:
    eval_models = []

In [ ]:
win_rates = {}
win_counts = {}
tie_counts = {}
loss_counts = {}
total_counts = {}

for model_name in eval_models:
    wins = 0
    ties = 0
    losses = 0
    total = 0
    
    for sample_id in common_sample_ids:
        if sample_id in baseline_data and sample_id in method_data:
            baseline_score = baseline_data[sample_id]['scores'].get(model_name)
            method_score = method_data[sample_id]['scores'].get(model_name)
            
            if baseline_score is not None and method_score is not None:
                total += 1
                if method_score > baseline_score:
                    wins += 1
                elif method_score == baseline_score:
                    ties += 1
                else:
                    losses += 1
    
    win_rate = wins / total if total > 0 else 0.0
    win_rates[model_name] = win_rate
    win_counts[model_name] = wins
    tie_counts[model_name] = ties
    loss_counts[model_name] = losses
    total_counts[model_name] = total
    
    print(f"{model_name}:")
    print(f"  Win rate: {win_rate:.4f} ({wins}/{total})")
    print(f"  Ties: {ties}, Losses: {losses}")
    print()


In [ ]:
# 以表格形式展示结果
import pandas as pd

results_df = pd.DataFrame({
    'Evaluation Model': list(win_rates.keys()),
    'Win Rate': [win_rates[model] for model in win_rates.keys()],
    'Wins': [win_counts[model] for model in win_rates.keys()],
    'Ties': [tie_counts[model] for model in win_rates.keys()],
    'Losses': [loss_counts[model] for model in win_rates.keys()],
    'Total': [total_counts[model] for model in win_rates.keys()],
})

results_df = results_df.sort_values('Win Rate', ascending=False)
print("\nWin Rates Summary:")
print(results_df.to_string(index=False))